In [25]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pandas as pd
import json
from functools import reduce
from datetime import datetime, timedelta
from glob import glob
from tqdm import tqdm
from copy import deepcopy
import shutil
def create_dir(dir):
    if not os.path.exists(dir):
        os.makedirs(dir)

In [2]:
def extract_keys(json_obj, parent_key=""):
    keys = []
    if isinstance(json_obj, dict):
        for key, value in json_obj.items():
            new_key = f"{parent_key}_{key}" if parent_key else key
            keys.append(new_key)
            keys.extend(extract_keys(value, new_key))  # 재귀 호출
    elif isinstance(json_obj, list):
        for i, item in enumerate(json_obj):
            new_key = f"{parent_key}[{i}]"
            keys.append(new_key)
            keys.extend(extract_keys(item, new_key))  # 리스트 항목 탐색
    return keys
src_label=pd.read_excel("../../data/raw/※ 근육주사_행위 및 구두 단위 분석_Time Check_Final.xlsx", sheet_name="Data_time")
with open('./video_time_match.json') as f:
    video_time_match = json.load(f)
video_time_match['필요한 물품 준비']=4
video_time_match['사용한 물품 정리']=29
key_list=list(video_time_match.keys())
for i in range(len(key_list)):
    create_dir(f"../../data/10sec_1/{key_list[i]}")
    create_dir(f"../../data/5sec_1/{key_list[i]}")
    create_dir(f"../../data/15sec_1/{key_list[i]}")

In [ ]:
video_path='../../data/raw/'
save_path='../../data/'
def to_seconds(dt):
    return dt.hour * 3600 + dt.minute * 60 + dt.second + dt.microsecond / 1e6
def read_all_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    original_fps=cap.get(cv2.CAP_PROP_FPS)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames, original_fps

def save_frames(frames, start_frame, end_frame, interval, save_path, class_name):
    create_dir(f"{save_path}/{class_name}")
    idx = 0
    for i in range(start_frame, end_frame, interval):
        if i >= len(frames):
            break
        frame = cv2.resize(frames[i], (512, 512))
        cv2.imwrite(f"{save_path}/{class_name}/{idx:04d}.png", frame)
        idx += 1
    return idx

def get_safe_time_range(target_time, end_time, fallback_seconds=5):
    # target_time이 음수 영역일 경우: fallback으로 0~5초 구간으로
    if target_time > end_time or target_time.date() != end_time.date():
        start_sec = 0
        end_sec = fallback_seconds
    else:
        start_sec = to_seconds(target_time)
        end_sec = to_seconds(end_time)
    return start_sec, end_sec


error_list=[]
for i in tqdm(range(200)):
    try:
        data_name = f'D{str(i+1).zfill(3)}'
        # 영상 로드 및 프레임 추출
        video_list_1 = glob(f'../../data/raw/{data_name}/1_*.mp4')
        video_list_2 = glob(f'../../data/raw/{data_name}/2_*.mp4')
        video_list_3 = glob(f'../../data/raw/{data_name}/3_*.mp4')

        frames_1, original_fps = read_all_frames(video_list_1[0])
        frames_2, _ = read_all_frames(video_list_2[0])
        frames_3, _ = read_all_frames(video_list_3[0])

        for j in range(len(key_list)):
            today = datetime.today().date()
            df = src_label.loc[video_time_match[key_list[j]]]
            time_obj = df[f'D{str(i+1)}']
            timestamp = datetime.combine(today, time_obj)
            frame_interval = int(original_fps / 5)
            # ===== 10초 전, 5fps =====
            target_time = timestamp - timedelta(seconds=10)
            start_sec, end_sec = get_safe_time_range(target_time, timestamp, fallback_seconds=10)

            start_frame = int(start_sec * original_fps)
            end_frame = int(end_sec * original_fps)

            save_path_10sec = f"../../data/10sec_1/{key_list[j]}/{data_name}"
            save_frames(frames_1, start_frame, end_frame, frame_interval, save_path_10sec, '1')
            save_frames(frames_2, start_frame, end_frame, frame_interval, save_path_10sec, '2')
            save_frames(frames_3, start_frame, end_frame, frame_interval, save_path_10sec, '3')

            # ===== 5초 전, 10fps =====
            target_time = timestamp - timedelta(seconds=5)
            start_sec, end_sec = get_safe_time_range(target_time, timestamp, fallback_seconds=5)
            start_frame = int(start_sec * original_fps)
            end_frame = int(end_sec * original_fps)

            save_path_5sec = f"../../data/5sec_1/{key_list[j]}/{data_name}"
            save_frames(frames_1, start_frame, end_frame, frame_interval, save_path_5sec, '1')
            save_frames(frames_2, start_frame, end_frame, frame_interval, save_path_5sec, '2')
            save_frames(frames_3, start_frame, end_frame, frame_interval, save_path_5sec, '3')
            
            target_time = timestamp - timedelta(seconds=15)
            start_sec, end_sec = get_safe_time_range(target_time, timestamp, fallback_seconds=15)
            start_sec = to_seconds(target_time)
            end_sec = to_seconds(timestamp)
            start_frame = int(start_sec * original_fps)
            end_frame = int(end_sec * original_fps)

            save_path_5sec = f"../../data/15sec_1/{key_list[j]}/{data_name}"
            save_frames(frames_1, start_frame, end_frame, frame_interval, save_path_5sec, '1')
            save_frames(frames_2, start_frame, end_frame, frame_interval, save_path_5sec, '2')
            save_frames(frames_3, start_frame, end_frame, frame_interval, save_path_5sec, '3')
            
    except:
        error_list.append(data_name)
        continue

 66%|██████▋   | 133/200 [3:00:21<1:28:59, 79.69s/it]

In [ ]:
def remove_empty_folders(path):
    for root, dirs, files in os.walk(path, topdown=False):
        for dir_name in dirs:
            dir_path = os.path.join(root, dir_name)
            # 폴더가 비어 있으면 삭제
            if not os.listdir(dir_path):
                os.rmdir(dir_path)
                print(f"Removed empty folder: {dir_path}")
            
folder_list=glob('../../data/*sec/**/**/**/')
for folder in folder_list:
    remove_empty_folders(folder)

In [27]:
for i in tqdm(range(len(folder_list))):
    dir_path1=glob(folder_list[i]+'**/')
    for dir_path in dir_path1:
        if not os.listdir(dir_path):
            shutil.rmtree(folder_list[i])
            print(f"Removed empty folder: {folder_list[i]}")
            break

 16%|█▋        | 1458/8865 [00:12<06:00, 20.57it/s] 

Removed empty folder: ../../data/5sec/물과 비누,알콜젤로 손위생 수행/D080/


 19%|█▊        | 1655/8865 [00:21<05:33, 21.59it/s]

Removed empty folder: ../../data/5sec/소독솜으로 닦고, 한손으로 주사바늘 뚜껑 제거/D080/


 21%|██        | 1851/8865 [00:30<05:50, 19.98it/s]

...

 68%|██████▊   | 6011/8865 [04:32<03:17, 14.44it/s]

Removed empty folder: ../../data/15sec/물과 비누로 손위생/D100/
Removed empty folder: ../../data/15sec/물과 비누로 손위생/D102/


 68%|██████▊   | 6014/8865 [04:32<02:09, 22.08it/s]

Removed empty folder: ../../data/15sec/물과 비누로 손위생/D104/


KeyboardInterrupt: 